# Imports

In [1]:
import cda2
#import datetime
import pyspark.sql.functions as F
import pyspark.sql.types as T
import json

from datetime import datetime, timedelta
from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number

# Connect to Spark

In [2]:
api = cda2.Api()

Set configuration parameters to better optimize queries.

In [3]:
config = {
    "spark.sql.adaptive.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.parallelismFirst": "false",
    "spark.sql.adaptive.coalescePartitions.minPartitionSize": "1m",
    "spark.executor.memory": "8g",
    "spark.executor.memoryOverhead": "16g",
}

Start Spark and specify number of cpus to use.

In [4]:
#api.start_spark(n_executors=100, config=config)
api.start_spark(n_executors=400)


:: loading settings :: file = /etc/spark/overlay/ivysettings.xml
:: loading settings :: url = jar:file:/usr/lib/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


http://looking-glass.cre-data-services/codev/ added as a remote repository with the name: repo-1
http://looking-glass.cre-data-services/central/ added as a remote repository with the name: repo-2
Ivy Default Cache set to: /home/rchong_mitre/.ivy2/cache
The jars for the packages stored in: /home/rchong_mitre/.ivy2/jars
org.mitre.spark#spark-geo_spark3.5_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-3864ba3c-a4d4-451e-a3c4-75c06cd8b71a;1.0
	confs: [default]
	found org.mitre.spark#spark-geo_spark3.5_2.12;0.2.0 in codev
	found org.mitre.spark#spark-geo-core_2.12;0.2.0 in codev
	found org.scala-lang.modules#scala-collection-compat_2.12;2.11.0 in central-proxy
	found net.sf.geographiclib#GeographicLib-Java;2.0 in central-proxy
	found org.ejml#ejml-core;0.43.1 in central-proxy
	found org.ejml#ejml-ddense;0.43.1 in central-proxy
	found com.esri.geometry#esri-geometry-api;2.2.4 in central-proxy
	found com.fasterxml.jackson.core#jackson-core;2.9.6 i

# Define global variables and functions

In [5]:
airports = [
    "KIAD",
]

In [6]:
%run ./shared_variables.ipynb
%run ./S3_csv_writers.ipynb

Function to convert Unix timestamp (milliseconds from 1970) to YYYYMMDD string.

In [7]:
#year0 = "2025"
year1 = str(int(year0) + 1)

In [8]:
dates = {"start_date": year0 + "-01-01", "end_date": year1 +"-01-01"}
print(dates)

{'start_date': '2026-01-01', 'end_date': '2027-01-01'}


In [9]:
print("retrieving procedures: ", datetime.now())

retrieving procedures:  2026-06-24 22:59:16.064480


# Retrieve SIDs and STARs for the airports of interest

<li>only get SID and STAR procedure types. (procedure_type = "APPROACH" is ignored)</li>
<li>create a unique <i>grouping</i> column.</li>
</br>
this will retrieve multiple versions of each fix, one for each update cycle in the year.

In [10]:
# v0 of this notebook didn't work when i tried it in oct 2025. this slack chat with matt pollock helped me figure out the issues...
# https://mitre.slack.com/archives/C2C46R03F/p1759873151426549

df_procedures = (
    api.dataframe("ArincTransition", **dates, metadata=True)
    .filter(F.col("airport.icao_region").startswith("K") | F.col("airport.icao_region").startswith("PA") | F.col("airport.icao_region").startswith("PH"))
#    .filter(F.col("procedure_name") == "CAVLR6")
#    .filter(F.col("procedure_name") == "DOCCS3")
#    .filter(F.col("procedure_name") == "GIBBZ5")
#    .filter(F.col("navigation_source") == "CIFP")
#    .filter(F.col("transition_name") == "ALL")
#    .filter(F.col("metadata.effective_end_date") == 1769040000000)
#    .filter(F.col("partitions").isNotNull())
    .withColumn("sorted_path_terminators", F.concat_ws(":", F.array_sort(F.split(F.col("path_terminators"), ":"))))      
#    .drop("path_terminators")
    .withColumn("end_date", F.col("metadata.effective_end_date"))
    .withColumn("transition_name", F.coalesce(F.col("transition_name"), F.lit("NONE")))
#    .withColumn("grouping", F.concat("navigation_source", F.lit("_"), "procedure_name", F.lit("_"), "transition_name", F.lit("_"), "sorted_path_terminators"))
    .withColumn("grouping", F.concat("navigation_source", F.lit("_"), "procedure_name", F.lit("_"), "transition_name", F.lit("_"), "path_terminators"))
    .drop("arinc_record_info", "arinc_route_type_qualifier_1", "arinc_route_type_qualifier_2", "arinc_route_type", "recommended_navaids", "alternative_dtpp_keys", "alternative_dtpp_procedure_names")
    .persist()
)

Could not find data for the following date ranges: 
    (2026-07-09, 2027-01-01)
Multiple versions found: 3.1.80, 3.1.81, 3.1.83, 3.1.84, 3.1.85, 3.1.86
                                                                                

In [11]:
df_procedures.count()

914461

In [12]:
# get the newest entry based on the end_date

window = Window.partitionBy("grouping").orderBy(col("end_date").desc())

df_procedures_newest = (df_procedures
    .withColumn("row", row_number().over(window))
    .filter(col("row") == 1)
    .drop("row", "end_date")
    .orderBy("grouping")
    .persist()
)

In [13]:
df_procedures_newest.count()

148691

In [14]:
#df_procedures_newest.select("path_terminators").show(df_procedures.count(), truncate=False)
#df_procedures_newest.filter(F.col("primary_key") == '4S5N2h7TvDA-0_L').select("legs").show(truncate=False)
#df_procedures_newest.show()

In [15]:
#df_procedures.select("path_terminators").show(truncate=False)
df_procedures_newest = df_procedures_newest.dropDuplicates(["navigation_source", "procedure_name", "transition_name", "sorted_path_terminators"]).orderBy(["navigation_source", "procedure_name", "transition_name", "sorted_path_terminators"])
#df_procedures_newest.show()

In [16]:
# hack to just use one navigation_source. if there's two, use the first; don't care which; if there's one, then just use it.

df_available_nav_sources = (
    df_procedures_newest
    .select(
        "procedure_name",
        "navigation_source",
    )
    .distinct()
    .orderBy("procedure_name")
)

window = Window.partitionBy("procedure_name").orderBy(col("navigation_source"))
df_available_nav_sources = (df_available_nav_sources
    .withColumn("row", row_number().over(window))
    .filter(col("row") == 1)
    .drop("row")
)

df_procedures_newest = df_procedures_newest.join(df_available_nav_sources, on=["procedure_name", "navigation_source"], how="left_semi")

In [17]:
df_procedures_newest.count()

75657

In [18]:
#df_procedures_newest.show()

In [19]:
# v0 of this notebook didn't work when i tried it in oct 2025. this slack chat with matt pollock helped me figure out the issues...
# https://mitre.slack.com/archives/C2C46R03F/p1759873151426549

df_procedures_exploded = (
    df_procedures_newest
    .select(
        F.col("primary_key").alias("segment_id"),
        "dtpp_procedure_name",
        "procedure_name",
        "transition_name",
        "procedure_type",
        "transition_type",
        "navigation_source",
        F.col("airport.icao_region").alias("airport_icao_region"),
#        F.col("airport.name").alias("airport"),
        F.explode("legs").alias("leg"),
#        F.col("metadata.effective_end_date").alias("end_date")
    )
    .withColumn("fix_name", F.col("leg.path_terminator.identification.name"))
    .withColumn("fix_icao_region", F.col("leg.path_terminator.identification.icao_region"))
    .withColumn("seq", F.col("leg.sequence_number").alias("seq"))
    .withColumn("latitude", F.col("leg.path_terminator.latitude"))
    .withColumn("longitude", F.col("leg.path_terminator.longitude"))
    .withColumn("magnetic_variation", F.col("leg.path_terminator.magnetic_variation.modeled"))
    .withColumn("speed_description", F.col("leg.speed_limit.descriptor"))
    .withColumn("speed_limit", F.col("leg.speed_limit.limit"))
    .withColumn("speed_altitude", F.col("leg.speed_limit.altitude"))
    .withColumn("altitude_description", F.col("leg.altitude_limits.description"))
    .withColumn("altitude_value1", F.col("leg.altitude_limits.value1"))
    .withColumn("altitude_value2", F.col("leg.altitude_limits.value2"))
#    .withColumn("transition_name", F.coalesce(F.col("transition_name"), F.lit("n/a")))
#    .withColumn("grouping", F.concat("procedure_name", F.lit("_"), "transition_name", F.lit("_"), F.format_string("%03d", F.col("seq")), F.lit("_"), "fix_name"))
#    .withColumn("grouping", F.concat("navigation_source", F.lit("_"), "procedure_name", F.lit("_"), "transition_name", F.lit("_"), F.format_string("%03d", F.col("seq")), F.lit("_"), "fix_name"))
    .drop("leg")
#    .dropDuplicates(["path_terminators"])
#    .orderBy("grouping", col("end_date").desc())
    .persist()
)

df_procedures_exploded = (
    df_procedures_exploded
    .filter(F.col("fix_name").isNotNull())
    .filter(F.col("latitude").isNotNull())
    .filter(F.col("longitude").isNotNull())
    .filter(F.col("magnetic_variation").isNotNull())
    .filter(F.col("procedure_type").isin("SID", "STAR"))
 
#    .filter(F.col("airport_icao_region").startswith("K") | F.col("airport_icao_region").startswith("PA") | F.col("airport_icao_region").startswith("PH"))

#    .filter(F.col("navigation_source").isin(["CIFP", "LIDO"]))
#    .filter(F.col("end_date") == 1769040000000)
#    .filter(F.col("navigation_source") == "CIFP")
#    .filter(F.col("navigation_source") == "LIDO")

#    .filter(F.col("procedure_name") == "LGTNG5")
#    .filter(F.col("procedure_name") == "BNFSH2")
#    .filter(F.col("procedure_name") == "ATL2")
#    .filter(F.col("procedure_name") == "SCOT7")
#    .filter(F.col("procedure_name") == "CAVLR6")
#    .filter(F.col("procedure_name") == "GIBBZ5")
#    .filter(F.col("procedure_name") == "SUNSS8")
#    .filter(F.col("procedure_name").isin(["GIBBZ5", "CAVLR6"]))

#    .filter(F.col("fix_name") == "MEEGO")

#    comment the following out so we get procedures for all US airports
#    .filter(F.col("airport").isin(airports))

#    .filter(F.col("grouping").isNotNull())
#    .filter(F.col("transition_name").isNotNull())
#    .filter(F.col("fix_name").isNotNull())
    .persist()
)

In [20]:
df_procedures_exploded.count()

42499

In [21]:
#df_procedures_exploded.show(df_procedures_exploded.count())

In [23]:
# df_procedures_exploded_output = (
#     df_procedures_exploded
#         .select(
#             "dtpp_procedure_name",
#             "navigation_source",
#             "procedure_name",
#             "procedure_type",
#             "transition_name",
#             "segment_id",
#             "transition_type",
#             "fix_name",
#             "fix_icao_region",
#             "seq",
#             "latitude",
#             "longitude",
#             "magnetic_variation",
# #            "airport",
#             "speed_description",
#             "speed_limit",
#             "speed_altitude",
#             "altitude_description",
#             "altitude_value1",
#             "altitude_value2",
#         )
# #        .coalesce(1)
#         .withColumn("proc_type", F.col("procedure_type"))
#         .withColumn("proc_name", F.col("procedure_name"))
#         .repartition("proc_type", "proc_name")
# )

In [24]:
# (
#     df_procedures_exploded_output
#         .repartition(1)
#         .write.option("header", True)
#         .partitionBy(["proc_type", "proc_name"])
#         .csv("CRAFT/" + year0 + "/procedures", compression="None", mode="overwrite")
# )

In [25]:
# spark_write_cre(df_procedures_exploded_output, path="CRAFT/" + year0 + "/procedures", writer=write_csv_with_partitions, partitionBy=["proc_type", "proc_name"], compression=None, mode="overwrite")

In [26]:
# (
#     df_procedures_exploded
#         .select(
#             "dtpp_procedure_name",
#             "navigation_source",
#             "procedure_name",
#             "procedure_type",
#             "transition_name",
#             "segment_id",
#             "transition_type",
#             "fix_name",
#             "fix_icao_region",
#             "seq",
#             "latitude",
#             "longitude",
#             "magnetic_variation",
# #            "airport",
#             "speed_description",
#             "speed_limit",
#             "speed_altitude",
#             "altitude_description",
#             "altitude_value1",
#             "altitude_value2",
#         )
#         .coalesce(1)
#         .write.option("header", True)
#         .csv("CRAFT/" + year0 + "/procedures/all_sidstar", compression="None", mode="overwrite")
# )

In [27]:
df_procedures_sidstar_output = (
    df_procedures_exploded
        .select(
            "dtpp_procedure_name",
            "navigation_source",
            "procedure_name",
            "procedure_type",
            "transition_name",
            "segment_id",
            "transition_type",
            "fix_name",
            "fix_icao_region",
            "seq",
            "latitude",
            "longitude",
            "magnetic_variation",
#            "airport",
            "speed_description",
            "speed_limit",
            "speed_altitude",
            "altitude_description",
            "altitude_value1",
            "altitude_value2",
        )
)

In [28]:
(
    df_procedures_sidstar_output
        .repartition(1)
        .write.option("header", True)
        .csv("CRAFT/" + year0 + "/procedures/all_sidstar", compression="None", mode="overwrite")
)

In [29]:
# spark_write_cre(df_procedures_sidstar_output, path="CRAFT/" + year0 + "/procedures/all_sidstar", writer=write_csv, compression=None, mode="overwrite")

## output the procedures (SIDs and STARs) for each airport

In [30]:
# v0 of this notebook didn't work when i tried it in oct 2025. this slack chat with matt pollock helped me figure out the issues...
# https://mitre.slack.com/archives/C2C46R03F/p1759873151426549

df_procedures_by_airports = (
    api.dataframe("ArincTransition", **dates, metadata=True)
    .select(
        "dtpp_procedure_name",
        "procedure_name",
        "procedure_type",
        "navigation_source",
        F.col("airport.icao_region").alias("airport_icao_region"),
        F.col("airport.name").alias("airport"),
        F.col("metadata.effective_end_date").alias("end_date")
    )
    .withColumn("grouping", F.concat("airport", F.lit("_"), "procedure_name"))
    .filter(F.col("airport_icao_region").startswith("K") | F.col("airport_icao_region").startswith("PA") | F.col("airport_icao_region").startswith("PH"))
    .filter(F.col("procedure_type").isin("SID", "STAR"))

#    comment the following out so we get procedures for all US airports
#    .filter(F.col("airport").isin(airports)) 
#    .filter(F.col("airport") == "KIAD")

    .orderBy("grouping")
    .persist()
)

Could not find data for the following date ranges: 
    (2026-07-09, 2027-01-01)
Multiple versions found: 3.1.80, 3.1.81, 3.1.83, 3.1.84, 3.1.85, 3.1.86
                                                                                

In [31]:
window = Window.partitionBy("grouping").orderBy(col("end_date").desc())

df_procedures_by_airports_newest = (df_procedures_by_airports
    .withColumn("row", row_number().over(window))
    .filter(col("row") == 1)
    .drop("row", "end_date")
    .orderBy("procedure_type", "dtpp_procedure_name", "grouping")
)

In [32]:
df_procedures_by_airports_newest.count()

4761

In [33]:
#df_procedures_by_airports_newest.show(df_procedures_by_airports_newest.count())
#df_procedures_by_airports_newest.show()

In [34]:
# (
#     df_procedures_by_airports_newest
#         .select(
#             "airport",
#             "procedure_type",
#             "procedure_name",
#         )
#         .orderBy("airport", "procedure_type", "procedure_name")
#         .write.option("header", True)
#         .csv("CRAFT/" + year0 + "/procedures/airports", compression="None", mode="overwrite")
# )

In [35]:
df_procedures_by_airports_newest_output = (
    df_procedures_by_airports_newest
        .select(
            "airport",
            "procedure_type",
            "procedure_name",
        )
        .orderBy("airport", "procedure_type", "procedure_name")
)

In [36]:
(
    df_procedures_by_airports_newest_output
        .repartition(1)
        .write.option("header", True)
        .csv("CRAFT/" + year0 + "/procedures/airports", compression="None", mode="overwrite")
)

In [37]:
# spark_write_cre(df_procedures_by_airports_newest_output, path="CRAFT/" + year0 + "/procedures/airports", writer=write_csv, compression=None, mode="overwrite")

## output all procedure fixes
<li>add a column that concats all the procedures that use the fix</li>
<li>add a column that concats the procedure types that use the fix</li>

In [38]:
#(
#    df_procedures_exploded.groupBy("fix_name", "fix_icao_region", "latitude", "longitude", "magnetic_variation")
#        .agg(F.concat_ws(":", F.collect_set("procedure_name")).alias("procedures_using_fix"), F.concat_ws(":", F.collect_set("procedure_type")).alias("procedure_types_using_fix"))
#        .orderBy("fix_name")
#        .write.option("header", True)
#        .csv("CRAFT/" + year0 + "/procedures/all_fixes", compression="None", mode="overwrite")
#)

In [39]:
df_procedures_allfixes_output = (
    df_procedures_exploded.groupBy("fix_name", "fix_icao_region", "latitude", "longitude", "magnetic_variation")
        .agg(F.concat_ws(":", F.collect_set("procedure_name")).alias("procedures_using_fix"), F.concat_ws(":", F.collect_set("procedure_type")).alias("procedure_types_using_fix"))
        .orderBy("fix_name")
)

In [40]:
(
    df_procedures_allfixes_output
        .repartition(1)
        .write.option("header", True)
        .csv("CRAFT/" + year0 + "/procedures/all_fixes", compression="None", mode="overwrite")
)

In [41]:
# spark_write_cre(df_procedures_allfixes_output, path="CRAFT/" + year0 + "/procedures/all_fixes", writer=write_csv, compression=None, mode="overwrite")